# Calcul de l'indicateur d'accessibilité aux médecins généralistes

Ce notebook combine :

- le voisinage routier (couples de communes à moins de 20 min et poids associés) ;
- l'offre et la demande communales calculées

L'indicateur est inspiré de l'**APL / E2SFCA** de la DREES.

**Attention :** il ne s'agit pas d'une reproduction exacte de l'APL officiel, car notre offre est mesurée par un nombre de médecins généralistes et non par leur activité réelle.

## 1. Principe

Pour chaque commune d'offre $j$, on calcule d'abord :

$$R_j = \frac{O_j}{\sum_i D_i \, w_{ij}}$$

avec :

- $O_j$ : nombre de médecins généralistes dans la commune $j$ ;
- $D_i$ : demande de la commune $i$ (population totale **ou** population standardisée selon l'âge) ;
- $w_{ij}$ : poids associé au temps routier entre $i$ et $j$.

Puis, pour chaque commune de résidence $i$ :

$$A_i = \sum_j R_j \, w_{ij}$$

On multiplie ensuite par 100 000 pour obtenir un nombre de médecins généralistes accessibles **pour 100 000 habitants**.

On calcule deux versions :

| Indicateur | Demande $D_i$ |
|---|---|
| `accessibilite_brute` | `population` |
| `accessibilite_ajustee_age` | `demande_standardisee` |

**Propriété de contrôle** : comme les mêmes poids sont utilisés aux deux étapes, on doit avoir
$\sum_i D_i A_i = \sum_j O_j$ : chaque médecin est « distribué » exactement une fois entre les habitants. On le vérifie en section 7.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)

## 2. Paramètres et chemins

- Entrées : sorties des personnes 1 et 2 (lecture en `https://` si le fichier est public, ou en `s3://`).
- Sortie : enregistrée en local puis, si `UPLOAD_MINIO = True`, envoyée sur notre bucket MinIO.

In [2]:
# --- Entrées (sorties des personnes 1 et 2, dans le bucket de Mariama) -------
CHEMIN_VOISINAGE = (
    "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/processed/"
    "voisinage_routier_20min.parquet"
)
CHEMIN_OFFRE_DEMANDE = (
    "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/processed/"
    "offre_demande_mg_commune.parquet"
)

# --- Sorties ----------------------------------------------------------------
NOM_FICHIER_SORTIE = "indicateur.parquet"

# Dossier personnel (toujours accessible en écriture, et hors du repo GitHub)
DOSSIER_SORTIE = Path.home() / "donnees_projet"
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
FICHIER_SORTIE_LOCAL = DOSSIER_SORTIE / NOM_FICHIER_SORTIE

UPLOAD_MINIO = True   # envoie le résultat sur ton stockage SSPCloud
CHEMIN_S3_SORTIE = f"emma457/Projet_python/{NOM_FICHIER_SORTIE}"

# --- Paramètres de l'indicateur ---------------------------------------------
ECHELLE = 100_000   # résultat exprimé pour 100 000 habitants

In [3]:
def systeme_fichiers_s3():
    # Connexion à MinIO avec les identifiants du service SSPCloud
    import s3fs
    return s3fs.S3FileSystem(
        client_kwargs={"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]}
    )


def lire_parquet(chemin):
    # Lit un parquet en local, en https (fichier public) ou en s3:// (MinIO)
    chemin = str(chemin)
    if chemin.startswith("s3://"):
        with systeme_fichiers_s3().open(chemin, "rb") as f:
            return pd.read_parquet(f)
    return pd.read_parquet(chemin)

## 3. Chargement des données

In [4]:
voisinage = lire_parquet(CHEMIN_VOISINAGE)
offre_demande = lire_parquet(CHEMIN_OFFRE_DEMANDE)

print("Voisinage :", voisinage.shape)
print("Offre-demande :", offre_demande.shape)

display(voisinage.head())
display(offre_demande.head())

Voisinage : (1082005, 9)
Offre-demande : (34858, 5)


,CODGEO_origine,LIBGEO_origine,CODGEO_destination,LIBGEO_destination,distance_vol_oiseau_km,distance_route_km,temps_route_min,classe_temps,poids_distance
0,01001,L'Abergement-Clémenciat,01001,L'Abergement-Clémenciat,0.000000,0.0000,0.000000,0-10 min,1.0
1,01001,L'Abergement-Clémenciat,01146,Dompierre-sur-Chalaronne,2.564389,5.2653,7.496667,0-10 min,1.0
2,01001,L'Abergement-Clémenciat,01412,Sulignat,3.257992,5.7110,8.463333,0-10 min,1.0
3,01001,L'Abergement-Clémenciat,01093,Châtillon-sur-Chalaronne,3.379304,6.8761,9.068333,0-10 min,1.0
4,01001,L'Abergement-Clémenciat,01188,Illiat,4.827127,7.2143,9.826667,0-10 min,1.0


,CODGEO,LIBGEO,population,demande_standardisee,nb_medecins_generalistes
0,01001,L'Abergement-Clémenciat,859.0,867.710799,0.00
1,01002,L'Abergement-de-Varey,273.0,270.430838,0.00
2,01004,Ambérieu-en-Bugey,15554.0,14951.854488,16.85
3,01005,Ambérieux-en-Dombes,1917.0,1861.977364,1.00
4,01006,Ambléon,114.0,124.706348,0.00


## 4. Contrôles et harmonisation des entrées

Avant de calculer, vérifions que les deux tables « se parlent » :

1. colonnes attendues présentes ;
2. codes communes au même format (texte sur 5 caractères) ;
3. pas de doublons ;
4. pas de valeurs manquantes ou négatives, poids compris entre 0 et 1 ;
5. aucun médecin ni habitant « perdu » parce que sa commune n'est pas dans le voisinage ;
6. chaque commune est voisine d'elle-même (temps 0, poids 1).

In [5]:
# 4.1 Colonnes nécessaires
colonnes_voisinage = {"CODGEO_origine", "CODGEO_destination", "poids_distance"}
colonnes_offre_demande = {
    "CODGEO", "LIBGEO", "population",
    "demande_standardisee", "nb_medecins_generalistes",
}

manquantes_voisinage = colonnes_voisinage - set(voisinage.columns)
manquantes_offre = colonnes_offre_demande - set(offre_demande.columns)

if manquantes_voisinage:
    raise ValueError(f"Colonnes manquantes dans le voisinage : {manquantes_voisinage}")
if manquantes_offre:
    raise ValueError(f"Colonnes manquantes dans offre-demande : {manquantes_offre}")

print("Colonnes nécessaires présentes.")

Colonnes nécessaires présentes.


In [6]:
# 4.2 Codes communes : texte sur 5 caractères ('1001' -> '01001', '1001.0' -> '01001', Corse 2A/2B conservée)
def normaliser_codgeo(serie):
    return (
        serie.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(5)
    )

for col in ["CODGEO_origine", "CODGEO_destination"]:
    voisinage[col] = normaliser_codgeo(voisinage[col])
offre_demande["CODGEO"] = normaliser_codgeo(offre_demande["CODGEO"])

print("Exemples de codes :", offre_demande["CODGEO"].sample(5, random_state=0).tolist())

Exemples de codes : ['01329', '59520', '12216', '27287', '69020']


In [7]:
# 4.3 Doublons
if offre_demande["CODGEO"].duplicated().any():
    raise ValueError("La table offre-demande contient des CODGEO dupliqués.")

n_doublons = voisinage.duplicated(["CODGEO_origine", "CODGEO_destination"]).sum()
print("Couples origine-destination en double :", n_doublons)
if n_doublons:
    voisinage = voisinage.drop_duplicates(["CODGEO_origine", "CODGEO_destination"])

print("Nombre de communes :", offre_demande["CODGEO"].nunique())
print("Nombre de couples origine-destination :", len(voisinage))

Couples origine-destination en double : 0
Nombre de communes : 34858
Nombre de couples origine-destination : 1082005


In [8]:
# 4.4 Valeurs manquantes, négatives, poids
cols_num = ["population", "demande_standardisee", "nb_medecins_generalistes"]

print("Valeurs manquantes :")
print(offre_demande[cols_num].isna().sum())
offre_demande[cols_num] = offre_demande[cols_num].fillna(0)

if (offre_demande[cols_num] < 0).any().any():
    raise ValueError("Valeurs négatives dans offre-demande.")

print(
    f"\nPoids : min = {voisinage['poids_distance'].min():.3f}, "
    f"max = {voisinage['poids_distance'].max():.3f}"
)
print("Répartition des poids :")
print(voisinage["poids_distance"].round(3).value_counts().sort_index())

if not voisinage["poids_distance"].between(0, 1).all():
    raise ValueError("Des poids sortent de [0, 1].")

# Un poids nul n'apporte rien au calcul
voisinage = voisinage[voisinage["poids_distance"] > 0].copy()

Valeurs manquantes :
population                  4
demande_standardisee        0
nb_medecins_generalistes    0
dtype: int64

Poids : min = 0.333, max = 1.000
Répartition des poids :
poids_distance
0.333    541094
0.667    332752
1.000    208159
Name: count, dtype: int64


In [9]:
# 4.5 Communes qui ne trouvent pas leur place dans le voisinage
offre_perdue = offre_demande[
    (offre_demande["nb_medecins_generalistes"] > 0)
    & ~offre_demande["CODGEO"].isin(voisinage["CODGEO_destination"])
]
population_perdue = offre_demande[
    (offre_demande["population"] > 0)
    & ~offre_demande["CODGEO"].isin(voisinage["CODGEO_origine"])
]
codes_voisinage_inconnus = (
    set(voisinage["CODGEO_origine"]) | set(voisinage["CODGEO_destination"])
) - set(offre_demande["CODGEO"])

print(
    f"Communes avec médecins absentes des destinations : {len(offre_perdue)} "
    f"({offre_perdue['nb_medecins_generalistes'].sum():.0f} médecins)"
)
print(
    f"Communes peuplées absentes des origines : {len(population_perdue)} "
    f"({population_perdue['population'].sum():,.0f} habitants)"
)
print(f"Codes du voisinage absents d'offre-demande : {len(codes_voisinage_inconnus)}")

# À regarder en priorité : arrondissements Paris / Lyon / Marseille, communes nouvelles
display(offre_perdue.sort_values("nb_medecins_generalistes", ascending=False).head(10))

Communes avec médecins absentes des destinations : 0 (0 médecins)
Communes peuplées absentes des origines : 0 (0 habitants)
Codes du voisinage absents d'offre-demande : 0


,CODGEO,LIBGEO,population,demande_standardisee,nb_medecins_generalistes


In [10]:
# 4.6 Chaque commune doit être voisine d'elle-même (temps 0 -> poids 1)
deja_presentes = set(
    voisinage.loc[
        voisinage["CODGEO_origine"] == voisinage["CODGEO_destination"],
        "CODGEO_origine",
    ]
)
a_ajouter = sorted(set(offre_demande["CODGEO"]) - deja_presentes)
print("Couples (i, i) ajoutés :", len(a_ajouter))

if a_ajouter:
    voisinage = pd.concat(
        [
            voisinage,
            pd.DataFrame({
                "CODGEO_origine": a_ajouter,
                "CODGEO_destination": a_ajouter,
                "poids_distance": 1.0,
            }),
        ],
        ignore_index=True,
    )

Couples (i, i) ajoutés : 0


In [11]:
# 4.7 Comparaison des deux demandes
total_pop = offre_demande["population"].sum()
total_std = offre_demande["demande_standardisee"].sum()
print(f"Population totale         : {total_pop:,.0f}")
print(f"Demande standardisée      : {total_std:,.0f}")
print(f"Rapport                   : {total_std / total_pop:.3f}")

Population totale         : 67,760,573
Demande standardisée      : 67,760,573
Rapport                   : 1.000


## 5. Fonction de calcul (E2SFCA)

Les deux étapes sont regroupées dans une fonction, pour pouvoir :

- calculer la version brute et la version ajustée avec le même code ;
- la réutiliser dans le notebook 04 avec d'autres poids (sensibilité).

In [12]:
def calcul_accessibilite(voisinage, offre_demande, col_demande,
                         col_offre="nb_medecins_generalistes",
                         col_poids="poids_distance", echelle=ECHELLE):
    # Renvoie :
    #   accessibilite : Series indexée par CODGEO (médecins accessibles pour `echelle` habitants)
    #   ratio         : Series indexée par CODGEO des communes d'offre (R_j)
    demande = offre_demande.set_index("CODGEO")[col_demande]
    offre = offre_demande.set_index("CODGEO")[col_offre]

    couples = voisinage[["CODGEO_origine", "CODGEO_destination", col_poids]].copy()

    # Étape 1 : demande pondérée qui pèse sur chaque commune d'offre j
    couples["demande_ponderee"] = (
        couples["CODGEO_origine"].map(demande).fillna(0) * couples[col_poids]
    )
    demande_accessible = couples.groupby("CODGEO_destination")["demande_ponderee"].sum()

    # Ratio offre / demande R_j
    offre_j = offre.reindex(demande_accessible.index).fillna(0)
    ratio = (offre_j / demande_accessible).where(demande_accessible > 0, 0.0)
    ratio = ratio[offre_j > 0]

    sans_demande = offre_j[(offre_j > 0) & (demande_accessible == 0)]
    if len(sans_demande):
        print(f"  Attention : {len(sans_demande)} communes ont des médecins mais "
              f"aucune demande accessible ({sans_demande.sum():.0f} médecins ignorés).")

    # Étape 2 : somme des R_j accessibles depuis chaque commune i
    couples["contribution"] = (
        couples["CODGEO_destination"].map(ratio).fillna(0) * couples[col_poids]
    )
    accessibilite = couples.groupby("CODGEO_origine")["contribution"].sum()

    accessibilite = accessibilite.reindex(offre_demande["CODGEO"]).fillna(0) * echelle
    return accessibilite, ratio

## 6. Calcul des deux versions

In [13]:
print("Version brute (population totale)")
acc_brute, ratio_brute = calcul_accessibilite(voisinage, offre_demande, "population")

print("Version ajustée par l'âge (demande standardisée)")
acc_age, ratio_age = calcul_accessibilite(voisinage, offre_demande, "demande_standardisee")

resultat = offre_demande[
    ["CODGEO", "LIBGEO", "population", "demande_standardisee", "nb_medecins_generalistes"]
].copy()

resultat["accessibilite_brute"] = resultat["CODGEO"].map(acc_brute).values
resultat["accessibilite_ajustee_age"] = resultat["CODGEO"].map(acc_age).values

# Densité « naïve » : médecins de la commune / habitants de la commune (pour comparaison)
resultat["densite_mg"] = np.where(
    resultat["population"] > 0,
    resultat["nb_medecins_generalistes"] / resultat["population"] * ECHELLE,
    np.nan,
)

# Effet de l'âge : < 1 = la commune perd en accessibilité quand on tient compte de l'âge
resultat["ratio_age_brute"] = np.where(
    resultat["accessibilite_brute"] > 0,
    resultat["accessibilite_ajustee_age"] / resultat["accessibilite_brute"],
    np.nan,
)

print("\nTable finale :", resultat.shape)
resultat.head()

Version brute (population totale)
Version ajustée par l'âge (demande standardisée)

Table finale : (34858, 9)


,CODGEO,LIBGEO,population,demande_standardisee,nb_medecins_generalistes,accessibilite_brute,accessibilite_ajustee_age,densite_mg,ratio_age_brute
0,01001,L'Abergement-Clémenciat,859.0,867.710799,0.00,54.852012,54.605783,0.000000,0.995511
1,01002,L'Abergement-de-Varey,273.0,270.430838,0.00,87.995423,89.250994,0.000000,1.014269
2,01004,Ambérieu-en-Bugey,15554.0,14951.854488,16.85,90.860793,93.074762,108.332262,1.024367
3,01005,Ambérieux-en-Dombes,1917.0,1861.977364,1.00,82.010883,83.293918,52.164841,1.015645
4,01006,Ambléon,114.0,124.706348,0.00,53.617321,51.704013,0.000000,0.964315


## 7. Contrôles du résultat

1. **Conservation** : $\sum_i D_i A_i = \sum_j O_j$ pour chaque version.
2. Pas de doublons, pas de valeurs négatives ou infinies.
3. Ordre de grandeur : la moyenne pondérée par la population doit être proche de la densité nationale.

In [14]:
def verifier_conservation(accessibilite, ratio, col_demande, nom):
    od = offre_demande.set_index("CODGEO")
    offre_distribuee = od["nb_medecins_generalistes"].reindex(ratio.index).sum()
    reconstruit = (od[col_demande] * accessibilite.reindex(od.index) / ECHELLE).sum()
    ecart = abs(reconstruit - offre_distribuee) / offre_distribuee
    print(f"{nom:<8} médecins distribués = {offre_distribuee:,.1f} | "
          f"somme D_i x A_i = {reconstruit:,.1f} | écart relatif = {ecart:.1e}")
    if ecart > 1e-6:
        raise ValueError("Conservation non respectée : problème dans le calcul.")

verifier_conservation(acc_brute, ratio_brute, "population", "brute")
verifier_conservation(acc_age, ratio_age, "demande_standardisee", "âge")

print("\nMédecins au total :", offre_demande["nb_medecins_generalistes"].sum())

brute    médecins distribués = 93,762.4 | somme D_i x A_i = 93,762.4 | écart relatif = 0.0e+00
âge      médecins distribués = 93,762.4 | somme D_i x A_i = 93,762.4 | écart relatif = 0.0e+00

Médecins au total : 93762.41666666667


In [15]:
cols_acc = ["accessibilite_brute", "accessibilite_ajustee_age"]

print("Doublons :", resultat["CODGEO"].duplicated().sum())
print("Valeurs négatives :", (resultat[cols_acc] < 0).sum().to_dict())
print("Valeurs infinies ou manquantes :", (~np.isfinite(resultat[cols_acc])).sum().to_dict())

peuplees = resultat[resultat["population"] > 0]
densite_nationale = (
    resultat["nb_medecins_generalistes"].sum() / resultat["population"].sum() * ECHELLE
)
moyenne_ponderee = np.average(peuplees["accessibilite_brute"], weights=peuplees["population"])
print(f"\nDensité nationale : {densite_nationale:.1f} MG pour 100 000 hab.")
print(f"Accessibilité brute moyenne (pondérée par la population) : {moyenne_ponderee:.1f}")

Doublons : 0
Valeurs négatives : {'accessibilite_brute': 0, 'accessibilite_ajustee_age': 0}
Valeurs infinies ou manquantes : {'accessibilite_brute': 0, 'accessibilite_ajustee_age': 0}

Densité nationale : 138.4 MG pour 100 000 hab.
Accessibilité brute moyenne (pondérée par la population) : 138.4


## 8. Premiers résultats

In [16]:
display(
    peuplees[cols_acc + ["densite_mg"]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
)

sans_medecin = peuplees[peuplees["nb_medecins_generalistes"] == 0]
print(f"Communes peuplées sans médecin : {len(sans_medecin):,}")
print(f"  dont accessibilité > 0 grâce aux voisines : "
      f"{(sans_medecin['accessibilite_brute'] > 0).sum():,}")
print(f"Communes peuplées avec accessibilité nulle : "
      f"{(peuplees['accessibilite_brute'] == 0).sum():,}")

print("\nCorrélation de rang (Spearman) entre densité simple et indicateur :",
      round(peuplees[["densite_mg", "accessibilite_brute"]].corr("spearman").iloc[0, 1], 3))

,accessibilite_brute,accessibilite_ajustee_age,densite_mg
count,34848.000000,34848.000000,34848.000000
mean,82.987329,79.341814,45.507211
std,55.454683,52.704693,141.789458
min,0.000000,0.000000,0.000000
10%,24.810289,23.516625,0.000000
25%,46.561817,44.180048,0.000000
50%,73.612898,70.423051,0.000000
75%,108.435730,103.396210,42.584917
90%,150.183956,144.020957,155.465636
max,1232.876712,1077.910810,9760.567764


Communes peuplées sans médecin : 25,441
  dont accessibilité > 0 grâce aux voisines : 24,520
Communes peuplées avec accessibilité nulle : 921

Corrélation de rang (Spearman) entre densité simple et indicateur : 0.331


In [17]:
print("20 communes les moins bien loties (version ajustée âge) :")
display(peuplees.nsmallest(20, "accessibilite_ajustee_age"))

print("10 communes les mieux loties :")
display(peuplees.nlargest(10, "accessibilite_ajustee_age"))

20 communes les moins bien loties (version ajustée âge) :


,CODGEO,LIBGEO,population,demande_standardisee,nb_medecins_generalistes,accessibilite_brute,accessibilite_ajustee_age,densite_mg,ratio_age_brute
58,01066,La Burbanche,97.0,93.903799,0.0,0.0,0.0,0.0,NaN
71,01080,Champdor-Corcelles,655.0,666.012582,0.0,0.0,0.0,0.0,NaN
117,01135,Crozet,2354.0,2124.529100,0.0,0.0,0.0,0.0,NaN
133,01153,Échenevex,2227.0,2000.230436,0.0,0.0,0.0,0.0,NaN
179,01210,Lélex,236.0,231.663914,0.0,0.0,0.0,0.0,NaN
184,01215,Surjoux-Lhopital,140.0,145.626559,0.0,0.0,0.0,0.0,NaN
443,02055,Beaumé,84.0,97.539438,0.0,0.0,0.0,0.0,NaN
1254,03067,Châtelperron,132.0,137.717876,0.0,0.0,0.0,0.0,NaN
1514,04016,Authon,59.0,63.128575,0.0,0.0,0.0,0.0,NaN
1518,04020,Barles,124.0,126.840594,0.0,0.0,0.0,0.0,NaN


10 communes les mieux loties :


,CODGEO,LIBGEO,population,demande_standardisee,nb_medecins_generalistes,accessibilite_brute,accessibilite_ajustee_age,densite_mg,ratio_age_brute
2277,07262,Saint-Laurent-les-Bains-Laval-d'Aurelle,184.0,212.132006,3.000000,1232.876712,1077.910810,1630.434783,0.874305
5540,17004,Île-d'Aix,196.0,266.555523,2.333333,1190.476190,875.364843,1190.476190,0.735306
17438,48147,Saint-Étienne-du-Valdonnez,641.0,631.399229,5.400000,842.433697,855.243363,842.433697,1.015206
25726,65388,Saint-Lary-Soulan,837.0,862.486765,7.000000,755.124056,731.578101,836.320191,0.968818
1721,05023,Briançon,10748.0,11020.672937,67.833333,586.940428,571.900670,631.125171,0.974376
1709,05009,Aspres-lès-Corps,93.0,115.880969,0.000000,668.934955,567.335086,0.000000,0.848117
1855,05174,Val-des-Prés,610.0,575.437440,0.000000,547.862247,534.377817,0.000000,0.975387
28749,73132,Hauteluce,741.0,758.826311,4.000000,539.811066,527.129851,539.811066,0.976508
25822,66004,Les Angles,584.0,606.085860,4.500000,548.112058,525.207171,770.547945,0.958211
1825,05142,Saint-Firmin,449.0,514.727402,4.000000,603.280113,510.330795,890.868597,0.845927


## 9. Déciles (pour les cartes et l'analyse descriptive)

Calculés uniquement sur les communes peuplées. Le rang (`rank(method="first")`) évite que les nombreuses valeurs identiques (zéros) fassent disparaître des classes.

In [18]:
for col, nom in [("accessibilite_brute", "decile_brute"),
                 ("accessibilite_ajustee_age", "decile_ajuste_age")]:
    resultat[nom] = pd.NA
    masque = resultat["population"] > 0
    resultat.loc[masque, nom] = (
        pd.qcut(resultat.loc[masque, col].rank(method="first"), q=10, labels=False) + 1
    )
    resultat[nom] = resultat[nom].astype("Int64")

resultat[["decile_brute", "decile_ajuste_age"]].value_counts().head()

decile_brute  decile_ajuste_age
1             1                    3367
10            10                   3210
2             2                    3108
3             3                    2836
9             9                    2761
Name: count, dtype: int64

## 10. Sauvegarde

In [19]:
resultat.to_parquet(FICHIER_SORTIE_LOCAL, index=False)
print("Fichier enregistré en local :", FICHIER_SORTIE_LOCAL)

if UPLOAD_MINIO:
    fs = systeme_fichiers_s3()
    fs.put(str(FICHIER_SORTIE_LOCAL), CHEMIN_S3_SORTIE)
    print("Upload terminé : s3://" + CHEMIN_S3_SORTIE)
else:
    print("Upload MinIO désactivé (UPLOAD_MINIO = False).")

Fichier enregistré en local : /home/onyxia/donnees_projet/indicateur.parquet


PermissionError: Access Denied.

## Limites méthodologiques

Cet indicateur est **inspiré de l'APL**, mais n'est pas l'APL officiel de la DREES.

Dans notre version :

- l'offre correspond au **nombre de médecins généralistes** ;
- la demande est soit la **population totale**, soit une **population standardisée selon l'âge** ;
- l'accessibilité spatiale utilise les **temps routiers** ;
- les poids sont ceux calculés par la personne 1 :
  - jusqu'à 10 min : 1 ;
  - 10 à 15 min : 2/3 ;
  - 15 à 20 min : 1/3 ;
  - au-delà : 0.

L'APL officiel tient notamment compte du niveau réel d'activité des professionnels (consultations et visites, en équivalents temps plein). Notre résultat doit donc être présenté comme un **indicateur simplifié inspiré de l'APL / E2SFCA**, et la comparaison avec l'APL portera surtout sur les **rangs** des communes plutôt que sur les niveaux.